# Lab 4 — Quantize + Fine-Tune (QLoRA)
**Day 1 Afternoon | ~75 minutes | Colab T4 GPU**

---

## ⚠️ Switch to T4 GPU First
```
Runtime → Change runtime type → T4 GPU → Save
Then re-run from Cell 0.
```
Verify with `!nvidia-smi` in the GPU check cell.

## What You Will Build
By the end of this lab you will have:
1. Benchmarked FP16 vs NF4 quantization — VRAM, speed, quality side by side
2. Attached LoRA adapters to a quantized model and seen ~0.7% trainable parameters
3. Run a minimal QLoRA fine-tuning loop
4. Saved the adapter (~10 MB) and understood why that matters for deployment
5. Tested the fine-tuned model and compared it to the base model

> **The key idea:** A quantized base model (INT4) + tiny LoRA adapter = fine-tuned capability
> at a fraction of the memory. One base model can serve many tasks via adapter swaps.

In [ ]:
%%capture
!pip install transformers torch accelerate bitsandbytes peft trl datasets
print('Done')

In [ ]:
# GPU check — must show a T4 (or better) before continuing
import subprocess
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print(f'✅ GPU: {result.stdout.strip()}')
else:
    print('⚠️  No GPU found.')
    print('   Runtime → Change runtime type → T4 GPU → Save → re-run from Cell 0')

---

## Part A — Quantization Benchmark (25 min)

We load the same model twice: once in FP16 (baseline) and once in NF4 (quantized).
Measure load time, VRAM, and tokens/sec. Post your numbers to the shared leaderboard.

**Model:** `Qwen/Qwen2.5-1.5B-Instruct`  
**Fallback** (if VRAM error): change to `Qwen/Qwen2.5-0.5B-Instruct`

In [ ]:
import torch, time
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'  # change to 0.5B if VRAM errors

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print('Loading FP16 baseline...')
t0          = time.time()
model_fp16  = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map='auto'
)
load_fp16   = time.time() - t0
vram_fp16   = torch.cuda.memory_allocated() / 1e9
print(f'FP16 loaded  — {load_fp16:.1f}s  {vram_fp16:.2f} GB VRAM')

In [ ]:
# Benchmark helper — runs n_runs inferences, returns avg tokens/sec
def benchmark(model, tokenizer, prompt, n_runs=3, max_new_tokens=80):
    msgs      = [{'role': 'user', 'content': prompt}]
    formatted = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs    = tokenizer(formatted, return_tensors='pt').to(model.device)
    in_len    = inputs['input_ids'].shape[1]
    times     = []
    for _ in range(n_runs):
        t = time.time()
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                  do_sample=False, pad_token_id=tokenizer.eos_token_id)
        times.append(time.time() - t)
    avg   = sum(times) / len(times)
    toks  = out.shape[1] - in_len
    text  = tokenizer.decode(out[0][in_len:], skip_special_tokens=True)
    return {'speed': toks / avg, 'time': avg, 'sample': text}

PROMPT = 'Explain quantization vs pruning in 3 bullet points.'
stats_fp16 = benchmark(model_fp16, tokenizer, PROMPT)
print(f'FP16: {stats_fp16["speed"]:.1f} tok/s')
print(f'Sample: {stats_fp16["sample"][:120]}...')

del model_fp16
torch.cuda.empty_cache()

In [ ]:
# Load NF4 quantized model
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',              # NormalFloat4 — optimal for LLM weight distributions
    bnb_4bit_compute_dtype=torch.bfloat16,  # compute in BF16, store in 4-bit
    bnb_4bit_use_double_quant=True           # quantise the quantisation constants too
)

print('Loading NF4 model...')
t0        = time.time()
model_nf4 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map='auto'
)
load_nf4  = time.time() - t0
vram_nf4  = torch.cuda.memory_allocated() / 1e9
print(f'NF4 loaded   — {load_nf4:.1f}s  {vram_nf4:.2f} GB VRAM')

In [ ]:
stats_nf4 = benchmark(model_nf4, tokenizer, PROMPT)

print('QUANTIZATION COMPARISON')
print(f'{"Metric":<25} {"FP16":>10} {"NF4":>10}')
print('-' * 47)
print(f'{"VRAM (GB)":<25} {vram_fp16:>10.2f} {vram_nf4:>10.2f}')
print(f'{"Tokens/sec":<25} {stats_fp16["speed"]:>10.1f} {stats_nf4["speed"]:>10.1f}')
print(f'{"Load time (s)":<25} {load_fp16:>10.1f} {load_nf4:>10.1f}')
print(f'{"Memory ratio":<25} {"baseline":>10} {f"{vram_fp16/vram_nf4:.1f}x smaller":>10}')
print()
print('Quality check (same prompt):')
print(f'  FP16: {stats_fp16["sample"][:100]}...')
print(f'  NF4 : {stats_nf4["sample"][:100]}...')
print()
print('📋 Post your numbers to the class leaderboard.')

---

## Part B — LoRA Adapters (25 min)

We attach LoRA to the NF4 model (= QLoRA), print the trainable parameter count,
run a short fine-tuning loop, then measure the saved adapter.

> **LoRA mechanics:** `W_new = W + B·A` where B and A are low-rank matrices.
> The base weights are frozen. Only ~0.7% of parameters are trained.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_nf4.config.use_cache = False
model_nf4 = prepare_model_for_kbit_training(model_nf4)

lora_config = LoraConfig(
    r=16,                      # rank — bottleneck dimension; higher = more capacity
    lora_alpha=32,              # scaling: typically 2× rank
    target_modules='all-linear',
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)

model_peft = get_peft_model(model_nf4, lora_config)
model_peft.print_trainable_parameters()
# Expected: trainable params ~0.7% of total

In [ ]:
# Build a small instruction dataset about LLM deployment concepts
from datasets import Dataset

training_data = [
    {'instruction': 'What is quantization?',
     'response': 'Quantization reduces weight precision (FP32→INT4), shrinking memory 4–8× with minimal quality loss. NF4 is optimal for LLMs.'},
    {'instruction': 'What is LoRA?',
     'response': 'LoRA freezes base weights and adds small trainable rank-decomposition matrices BA. Only ~0.7% of parameters are trained.'},
    {'instruction': 'What is RAG?',
     'response': 'RAG retrieves relevant documents at inference time and injects them into the prompt, grounding the model in external knowledge.'},
    {'instruction': 'What is vLLM?',
     'response': 'vLLM is a serving engine using PagedAttention for efficient KV-cache management, enabling 20–30× throughput over naive serving.'},
    {'instruction': 'What is the difference between fine-tuning and RAG?',
     'response': 'Fine-tuning bakes behavior into weights (good for style/format). RAG retrieves at runtime (good for factual accuracy and updatable knowledge).'},
    {'instruction': 'What is PagedAttention?',
     'response': 'PagedAttention stores KV cache in non-contiguous memory pages, eliminating fragmentation and enabling efficient multi-user serving.'},
    {'instruction': 'What is QLoRA?',
     'response': 'QLoRA combines a 4-bit NF4 base model with 16-bit LoRA adapters, enabling 7B+ fine-tuning on a single consumer GPU.'},
    {'instruction': 'What is a context window?',
     'response': 'The maximum tokens a model processes in one pass — both input and output. Overflowing it causes truncation or errors.'},
    {'instruction': 'What is chunking in RAG?',
     'response': 'Chunking splits documents into segments before embedding. Smaller chunks improve precision; larger chunks preserve context.'},
    {'instruction': 'What is an embedding?',
     'response': 'A dense vector encoding semantic meaning. Similar texts cluster nearby in embedding space, enabling similarity search.'}
]

def format_example(ex):
    return {'text': tokenizer.apply_chat_template(
        [{'role': 'user',      'content': ex['instruction']},
         {'role': 'assistant', 'content': ex['response']}],
        tokenize=False, add_generation_prompt=False
    )}

dataset = Dataset.from_list(training_data).map(format_example)
print(f'Dataset: {len(dataset)} examples')
print(f'\nSample formatted example:')
print(dataset[0]['text'])

In [ ]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir='./lora_output',
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=True, fp16=False,
    logging_steps=5,
    save_strategy='no',
    report_to='none',
    max_length=512,
)

trainer = SFTTrainer(model=model_peft, train_dataset=dataset, args=training_args)

print('Starting QLoRA fine-tuning (~3–5 min on T4 for 10 examples × 3 epochs)...')
trainer.train()
print('✅ Training complete!')

---

## Part C — Save, Inspect, and Reload (25 min)

The adapter reveal: how big is the thing you actually ship?

In [ ]:
import os

adapter_path = './my_lora_adapter'
model_peft.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print('ADAPTER PACKAGE FILES')
total = 0
for f in os.listdir(adapter_path):
    size = os.path.getsize(os.path.join(adapter_path, f))
    total += size
    print(f'  {f}: {size/1e6:.1f} MB')

print(f'\nAdapter package total : {total/1e6:.1f} MB')
print(f'Base model            : ~{vram_nf4*1000:.0f} MB')
print()
print('→ You ship the adapter package (tiny). Users download the base model once.')
print('→ One base model can serve many tasks by swapping adapters.')

model_card = f"""---
base_model: {MODEL_ID}
library_name: peft
tags: [qlora, fine-tuned, quantization, llm-deployment]
---

# My QLoRA Fine-Tuned Adapter

- **Base model**: `{MODEL_ID}`
- **Adapter package size**: {total/1e6:.1f} MB (vs ~{vram_nf4*1000:.0f} MB quantized base in this runtime)
- **Trainable params**: ~0.7% of model parameters
- **Method**: QLoRA (NF4 + LoRA r=16)
- **Training**: 10 examples × 3 epochs on LLM deployment concepts

## Usage

```python
from peft import PeftModel

model = PeftModel.from_pretrained(base, "{adapter_path}")
```
"""
with open(f'{adapter_path}/README.md', 'w') as f:
    f.write(model_card)
print('✅ Tokenizer and model card written — adapter package is ready for a HuggingFace Hub push.')


In [ ]:
# Reload base + adapter and test the fine-tuned behavior
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map='auto')

def chat_with(model, q, max_new_tokens=150, do_sample=True):
    msgs      = [{'role': 'user', 'content': q}]
    formatted = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs    = tokenizer(formatted, return_tensors='pt').to(model.device)
    gen_kwargs = {
        'max_new_tokens': max_new_tokens,
        'do_sample': do_sample,
        'pad_token_id': tokenizer.eos_token_id,
    }
    if do_sample:
        gen_kwargs['temperature'] = 0.7
    with torch.no_grad():
        out = model.generate(**inputs, **gen_kwargs)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

def base_chat(q):
    return chat_with(base, q, do_sample=False)

comparison_questions = [
    'What is QLoRA?',
    'How does LoRA reduce trainable parameters?',
    'What is NF4 quantization?'
]
base_answers = {q: base_chat(q) for q in comparison_questions}

tuned = PeftModel.from_pretrained(base, adapter_path)

def tuned_chat(q):
    return chat_with(tuned, q, do_sample=True)

test_q = 'What is QLoRA?'
print(f'Q: {test_q}')
print(f'A: {tuned_chat(test_q)}')


### Before/After Comparison — The "I Changed the Model" Moment

The adapter is tiny, but it changes behavior. Run the table below to compare the base model against the fine-tuned adapter on the same prompts.


In [ ]:
# Before/After comparison — this is your "wow" cell
print(f'{"Question":<45} {"Base Model":<60} {"Fine-tuned"}')
print('-' * 145)
for q in comparison_questions:
    base_ans = base_answers[q][:80].replace('\n', ' ')
    tuned_ans = tuned_chat(q)[:80].replace('\n', ' ')
    print(f'{q:<45} {base_ans:<60} {tuned_ans}')


### Merge for Deployment Export

Adapters are great for serving many tasks from one base model. A merged model is useful when the target runtime expects one complete model directory, such as a GGUF/Ollama conversion flow.

For export, merge the adapter into an FP16 base model. The NF4 model was perfect for training cheaply; the merged artifact is what you prepare for conversion.

> In production, keep both artifacts when possible: the small adapter for flexible serving, and the merged model for export-oriented runtimes.


In [ ]:
# Merge adapter weights into an FP16 base model for export workflows.
# This merged directory is what you would convert to GGUF for Ollama or LM Studio.
try:
    merge_base = model_fp16
    print('Using the FP16 baseline model already loaded in Part A.')
except NameError:
    merge_base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16, device_map='auto'
    )

merge_peft = PeftModel.from_pretrained(merge_base, adapter_path)
merged_model = merge_peft.merge_and_unload()
merged_path = './my_merged_model'
merged_model.save_pretrained(merged_path)
tokenizer.save_pretrained(merged_path)
print(f'Merged model saved to {merged_path}')
print('This is the directory you would convert to GGUF for Ollama or LM Studio.')


In [ ]:
# What happens after this lab (outside Colab):
# 1. Export merged model:  merged_model.save_pretrained('./my_merged_model')
# 2. Convert to GGUF:      python convert_hf_to_gguf.py ./my_merged_model --outtype q4_k_m
# 3. Run in Ollama:        ollama create my-qlora -f Modelfile && ollama run my-qlora

print("""
DEPLOYMENT PATH:
  This Lab → merge_and_unload() → GGUF conversion → Ollama/LM Studio

  You just prepared the merged model directory. The GGUF conversion needs
  llama.cpp locally, but you now know exactly what artifact to export.
""")


---

## Bonus: Pruning in 5 Minutes

Pruning zeros out weights below a threshold. It sounds good in theory — but
sparse weights only accelerate on specialised hardware. For transformers in 2026,
**quantization almost always wins** over pruning for deployment.

The *practical* sparsity story is **Mixture of Experts (MoE)**: each token activates
only a subset of 'expert' FFN layers. The model is large in memory but cheap to run.

In [ ]:
# Pruning demo: zero out 30% of weights, measure quality impact
import torch.nn.utils.prune as prune_utils

# Use the small model from earlier (reload if needed)
demo_model = AutoModelForCausalLM.from_pretrained(
    'Qwen/Qwen2.5-0.5B-Instruct', torch_dtype=torch.bfloat16
)
demo_tok = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct')

# Count zero weights before
def sparsity(m):
    total = nonzero = 0
    for p in m.parameters():
        total   += p.numel()
        nonzero += p.nonzero().shape[0]
    return 1 - nonzero / total

print(f'Sparsity before pruning: {sparsity(demo_model):.1%}')

# Prune all Linear layers to 30% sparsity (global magnitude pruning)
params_to_prune = [
    (m, 'weight') for m in demo_model.modules()
    if isinstance(m, torch.nn.Linear)
]
prune_utils.global_unstructured(params_to_prune, pruning_method=prune_utils.L1Unstructured, amount=0.3)
for m, _ in params_to_prune:
    prune_utils.remove(m, 'weight')

print(f'Sparsity after  pruning: {sparsity(demo_model):.1%}')
print()

# Quick quality check
def quick_gen(m, t, prompt, max_new_tokens=40):
    msgs = [{'role': 'user', 'content': prompt}]
    fmt  = t.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp  = t(fmt, return_tensors='pt')
    ilen = inp['input_ids'].shape[1]
    with torch.no_grad():
        out = m.generate(**inp, max_new_tokens=max_new_tokens, do_sample=False,
                          pad_token_id=t.eos_token_id)
    return t.decode(out[0][ilen:], skip_special_tokens=True)

print('After 30% magnitude pruning:')
print(quick_gen(demo_model, demo_tok, 'What is quantization? One sentence.'))
print()
print('Key insight: 30% of weights zeroed → model size on disk unchanged (dense format).')
print('To realise size savings you need sparse kernels or a format like GGUF.')
print('For most LLM deployments in 2026: quantise first, prune rarely.')

In [ ]:
# PRUNING VERDICT — the most important thing to remember
print("""
╔══════════════════════════════════════════════════════════╗
║  PRUNING vs QUANTIZATION SCORECARD (2026)                ║
║                                                          ║
║  Pruning (this demo):                                    ║
║    ✅ Zeros out 30% of weights                           ║
║    ❌ File size unchanged (dense format)                  ║
║    ❌ No speedup without sparse CUDA kernels             ║
║    ❌ Quality degrades visibly                           ║
║                                                          ║
║  Quantization (what you did earlier):                    ║
║    ✅ Real memory savings (FP16 → NF4 = ~4x smaller)     ║
║    ✅ Works on every GPU today                           ║
║    ✅ Quality near-parity at 4-bit                       ║
║                                                          ║
║  → For LLM deployment in 2026: quantize first, prune ❌  ║
╚══════════════════════════════════════════════════════════╝
""")


---

## ✅ Lab 4 Complete — You Can Now:

- [ ] Explain WHY NF4 uses less VRAM than FP16 (not just that it does)
- [ ] Read `print_trainable_parameters()` and know what ~0.7% means for training cost
- [ ] Save an adapter + tokenizer + model card as a deployable package
- [ ] Load a base model + adapter and run inference (the real serving pattern)
- [ ] Compare base vs fine-tuned answers and explain what changed
- [ ] Merge an adapter into the base model for export workflows
- [ ] Explain why pruning doesn't shrink file size without sparse kernels
- [ ] Describe the path from this adapter → GGUF → Ollama in 3 steps

## 🚀 Stretch: Push to HuggingFace Hub

```bash
huggingface-cli login
```

```python
model_peft.push_to_hub("your-username/my-qlora-adapter")
tokenizer.push_to_hub("your-username/my-qlora-adapter")
# Your adapter is now live at huggingface.co/your-username/my-qlora-adapter
```

## Extra Stretch Goals

1. **Rank comparison:** Change `r=4` in the LoRA config, retrain, compare adapter size and answer quality vs `r=16`.
2. **Serving benchmark:** Compare inference speed for base + adapter vs the merged model.
3. **GGUF export:** Use `llama.cpp` locally to convert `./my_merged_model` to GGUF and run it in Ollama or LM Studio.
